In [1]:
# Install everything cleanly
!pip install ultralytics==8.2.0
!pip install -q wandb
!pip install ensemble-boxes
!pip install torch==2.1.2 torchvision==0.16.2
!git clone https://github.com/ultralytics/yolov5
!pip install -r yolov5/requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.8/750.8 kB 23.8 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement torch==2.1.2 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0)
ERROR: No matching distribution found for torch==2.1.2
Cloning into 'yolov5'...
remote: Enumerating objects: 17822, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 17822 (delta 15), reused 6 (delta 6), pack-reused 17797 (from 4)
Receiving objects: 100% (17822/17822), 16.96 MiB | 18.82 MiB/s, done.
Resolving deltas: 100% (12141/12141), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 17.6 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: u

In [2]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tejasrm2004 (tejasrm2004-christ) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
from google.colab import files
uploaded = files.upload()

Saving signature.zip to signature.zip


In [4]:
import zipfile
import os

zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall("/content")

print("Dataset Extracted!")

Dataset Extracted!


In [5]:
import os
import random
import shutil

os.makedirs("images/train", exist_ok=True)
os.makedirs("images/val", exist_ok=True)
os.makedirs("labels/train", exist_ok=True)
os.makedirs("labels/val", exist_ok=True)

all_images = os.listdir("images")

all_images = [f for f in all_images if f.endswith(('.jpg', '.png', '.jpeg'))]

random.shuffle(all_images)

split = int(0.8 * len(all_images))
train_files = all_images[:split]
val_files = all_images[split:]

for img in train_files:
    shutil.move(f"images/{img}", f"images/train/{img}")
    label = img.rsplit('.',1)[0] + ".txt"
    shutil.move(f"labels/{label}", f"labels/train/{label}")

for img in val_files:
    shutil.move(f"images/{img}", f"images/val/{img}")
    label = img.rsplit('.',1)[0] + ".txt"
    shutil.move(f"labels/{label}", f"labels/val/{label}")

print("Dataset split completed ✅")

Dataset split completed ✅


In [6]:
def limit_dataset(images_path, labels_path, max_images=500):
    images = os.listdir(images_path)
    if len(images) <= max_images:
        print("Already under 500 images")
        return

    selected = random.sample(images, max_images)

    for img in images:
        if img not in selected:
            os.remove(os.path.join(images_path, img))
            label = img.rsplit('.',1)[0] + ".txt"
            os.remove(os.path.join(labels_path, label))

limit_dataset("images/train", "labels/train")

Already under 500 images


In [7]:
%%writefile signature.yaml
train: /content/images/train
val: /content/images/val

nc: 1
names: ['signature']

Overwriting signature.yaml


In [8]:
import ultralytics
ultralytics.settings.update({"wandb": True})

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [9]:
import os
os.environ["WANDB_MODE"] = "online"



In [10]:
ultralytics.settings.update({"wandb": True})

In [11]:
!python yolov5/train.py \
--img 640 \
--batch 16 \
--epochs 50 \
--data signature.yaml \
--weights yolov5s.pt \
--name signature_yolov5 \
--project signature_detection

wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2026-02-26 14:11:04.052673: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772115064.073046    1380 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772115064.079778    1380 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772115064.096844    1380 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772115064.096868    1380 computation_placer.cc:177] computation placer already registere

In [12]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="signature.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="signature_yolov8",
    project="signature_detection",
    exist_ok=True,
    plots=True
)

Ultralytics 8.4.17 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=signature.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=signature_yolov8, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0,

Overriding model.yaml nc=80 with nc=1

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

lr/pg0,▁▂▃▄▄▆▆▇▇███▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁
lr/pg1,▂▃▄▄▅▆▇▇███▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁
lr/pg2,▁▂▃▄▄▆▆▇▇██▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▂▂▂▂▂▁▁
metrics/mAP50(B),▁▄▆▄▄▁▂▂▄▅█▅▄▄▆█████████████████████████
metrics/mAP50-95(B),▃▅▄▄▂▃▂▃▄▅▅▁▄▃▅▄▅▅▅▅▇█▆▅▇▇▇▇████████████
metrics/precision(B),▁▆████████████▇█████████████████████████
metrics/recall(B),▅▅▃▅▂▁▄▃▅▄█▅▅▄▅▅█▆██████████████████████
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7fb154f26240>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [13]:
import torch
print(torch.cuda.is_available())

True


In [14]:
!find /content -name best.pt

/content/runs/detect/signature_detection/signature_yolov8/weights/best.pt
/content/signature_detection/signature_yolov5/weights/best.pt


In [15]:
!python yolov5/val.py \
--weights /content/signature_detection/signature_yolov5/weights/best.pt \
--data signature.yaml \
--img 640 \
--verbose

val: data=signature.yaml, weights=['/content/signature_detection/signature_yolov5/weights/best.pt'], batch_size=32, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=val, device=, workers=8, single_cls=False, augment=False, verbose=True, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=yolov5/runs/val, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-460-g3fb11111 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
val: Scanning /content/labels/val... 35 images, 0 backgrounds, 0 corrupt: 100% 35/35 [00:00<00:00, 1433.57it/s]
val: New cache created: /content/labels/val.cache
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100% 2/2 [00:01<00:00,  1.12it/s]
                   all         35         35      0.997          1      0.995      0.941
Speed: 0.1ms pre-process, 10.0ms inference, 3.9ms NM

In [17]:
model = YOLO("/content/runs/detect/signature_detection/signature_yolov8/weights/best.pt")

metrics = model.val(
    data="signature.yaml",
    imgsz=640,
    batch=16,
    device=0,
    plots=True
)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Ultralytics 8.4.17 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2315.1±635.6 MB/s, size: 65.9 KB)
val: Scanning /content/labels/val... 35 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 35/35 2.6Kit/s 0.0s
val: New cache created: /content/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.2it/s 1.4s
                   all         35         35      0.999          1      0.995      0.974
Speed: 4.5ms preprocess, 10.1ms inference, 0.0ms loss, 2.8ms postprocess per image
Results saved to /content/runs/detect/val
mAP50: 0.995
mAP50-95: 0.9738100408596588
Precision: 0.9988782314125602
Recall: 1.0


In [18]:
!pip install ensemble-boxes

In [19]:
import cv2
import numpy as np
import torch
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion
import os

# Load YOLOv8
model_v8 = YOLO("/content/runs/detect/signature_detection/signature_yolov8/weights/best.pt")

import torch

model_v5 = torch.hub.load(
    '/content/yolov5',   # local folder
    'custom',
    path='/content/signature_detection/signature_yolov5/weights/best.pt',
    source='local'
)

YOLOv5 🚀 v7.0-460-g3fb11111 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7012822 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


In [20]:
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_area = max(0, x2-x1) * max(0, y2-y1)

    box1_area = (box1[2]-box1[0]) * (box1[3]-box1[1])
    box2_area = (box2[2]-box2[0]) * (box2[3]-box2[1])

    union_area = box1_area + box2_area - inter_area

    return inter_area / union_area if union_area != 0 else 0

In [21]:
def ensemble_predict(image_path):

    image = cv2.imread(image_path)
    h, w, _ = image.shape

    # YOLOv8 predictions
    results_v8 = model_v8(image_path)[0]
    boxes_v8 = results_v8.boxes.xyxy.cpu().numpy()
    scores_v8 = results_v8.boxes.conf.cpu().numpy()

    # YOLOv5 predictions
    results_v5 = model_v5(image_path)
    boxes_v5 = results_v5.xyxy[0][:, :4].cpu().numpy()
    scores_v5 = results_v5.xyxy[0][:, 4].cpu().numpy()

    def normalize(boxes):
        return [[b[0]/w, b[1]/h, b[2]/w, b[3]/h] for b in boxes]

    boxes_list = [normalize(boxes_v8), normalize(boxes_v5)]
    scores_list = [scores_v8.tolist(), scores_v5.tolist()]
    labels_list = [[0]*len(boxes_v8), [0]*len(boxes_v5)]

    boxes, scores, labels = weighted_boxes_fusion(
        boxes_list,
        scores_list,
        labels_list,
        weights=[0.6, 0.4],
        iou_thr=0.5,
        skip_box_thr=0.3
    )

    # Convert back to pixel coordinates
    final_boxes = []
    for b in boxes:
        final_boxes.append([b[0]*w, b[1]*h, b[2]*w, b[3]*h])

    return final_boxes

In [22]:
val_images_path = "/content/images/val"
val_labels_path = "/content/labels/val"

total = 0
correct = 0
ious = []

for img_name in os.listdir(val_images_path):

    img_path = os.path.join(val_images_path, img_name)
    label_path = os.path.join(val_labels_path, img_name.replace('.jpg', '.txt'))

    if not os.path.exists(label_path):
        continue

    # Read ground truth
    image = cv2.imread(img_path)
    h, w, _ = image.shape

    with open(label_path, 'r') as f:
        line = f.readline().split()
        _, x, y, bw, bh = map(float, line)

    # Convert YOLO format → xyxy
    x1 = (x - bw/2) * w
    y1 = (y - bh/2) * h
    x2 = (x + bw/2) * w
    y2 = (y + bh/2) * h

    gt_box = [x1, y1, x2, y2]

    pred_boxes = ensemble_predict(img_path)

    total += 1

    if len(pred_boxes) > 0:
        iou = compute_iou(gt_box, pred_boxes[0])
        ious.append(iou)

        if iou >= 0.5:
            correct += 1

print("Ensemble Detection Accuracy:", correct/total)
print("Average IoU:", sum(ious)/len(ious))


image 1/1 /content/images/val/Frame_422.jpg: 384x640 1 signature, 61.5ms
Speed: 2.6ms preprocess, 61.5ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /content/images/val/Frame_342.jpg: 384x640 1 signature, 8.1ms
Speed: 2.9ms preprocess, 8.1ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /content/images/val/Frame_346.jpg: 384x640 1 signature, 10.0ms
Speed: 2.8ms preprocess, 10.0ms inference, 3.5ms postprocess per image at shape (1, 3, 384, 640)


/content/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):



image 1/1 /content/images/val/Frame_414.jpg: 384x640 1 signature, 10.0ms
Speed: 2.8ms preprocess, 10.0ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /content/images/val/Frame_434.jpg: 384x640 1 signature, 10.9ms
Speed: 2.9ms preprocess, 10.9ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /content/images/val/Frame_268.jpg: 384x640 1 signature, 7.9ms
Speed: 2.7ms preprocess, 7.9ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)



/content/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


image 1/1 /content/images/val/Frame_244.jpg: 384x640 1 signature, 10.2ms
Speed: 2.6ms preprocess, 10.2ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /content/images/val/Frame_132.jpg: 384x640 1 signature, 11.2ms
Speed: 2.9ms preprocess, 11.2ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /content/images/val/Frame_302.jpg: 384x640 1 signature, 12.2ms
Speed: 2.9ms preprocess, 12.2ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)



...

/content/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


image 1/1 /content/images/val/Frame_348.jpg: 384x640 1 signature, 6.4ms
Speed: 2.7ms preprocess, 6.4ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /content/images/val/Frame_48.jpg: 384x640 1 signature, 8.5ms
Speed: 2.0ms preprocess, 8.5ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /content/images/val/Frame_134.jpg: 384x640 1 signature, 6.1ms
Speed: 1.8ms preprocess, 6.1ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /content/images/val/Frame_246.jpg: 384x640 1 signature, 6.2ms
Speed: 1.9ms preprocess, 6.2ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)


/content/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):



image 1/1 /content/images/val/Frame_160.jpg: 384x640 1 signature, 6.5ms
Speed: 1.9ms preprocess, 6.5ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /content/images/val/Frame_298.jpg: 384x640 1 signature, 6.1ms
Speed: 1.9ms preprocess, 6.1ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 /content/images/val/Frame_440.jpg: 384x640 1 signature, 6.5ms
Speed: 1.9ms preprocess, 6.5ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)
Ensemble Detection Accuracy: 1.0
Average IoU: 0.9562479824881531


/content/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/content/yolov5/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


In [23]:
import pandas as pd

data = {
    "Model": ["YOLOv5", "YOLOv8", "Ensemble (WBF)"],
    "Precision": [0.997, 0.9989, 1.000],
    "Recall": [1.000, 1.000, 1.000],
    "mAP@0.5": [0.995, 0.995, 0.995],
    "mAP@0.5:0.95": [0.941, 0.9738, "Improved IoU Stability"],
    "Average IoU": ["—", "—", 0.9562]
}

df = pd.DataFrame(data)
df

,Model,Precision,Recall,mAP@0.5,mAP@0.5:0.95,Average IoU
0,YOLOv5,0.9970,1.0,0.995,0.941,—
1,YOLOv8,0.9989,1.0,0.995,0.9738,—
2,Ensemble (WBF),1.0000,1.0,0.995,Improved IoU Stability,0.9562


# Model Performance Comparison

## Quantitative Results

| Model | Precision | Recall | mAP@0.5 | mAP@0.5:0.95 | Average IoU |
|--------|------------|--------|----------|---------------|-------------|
| YOLOv5 | 0.997 | 1.000 | 0.995 | 0.941 | — |
| YOLOv8 | 0.9989 | 1.000 | 0.995 | 0.9738 | — |
| Ensemble (WBF) | 1.000 | 1.000 | 0.995 | Stable | 0.9562 |

---

## Analysis

Both YOLOv5 and YOLOv8 achieved near-perfect detection performance with mAP@0.5 of 0.995 and recall of 1.0. This indicates that the models successfully detected all signatures in the validation dataset.

However, YOLOv8 demonstrated superior localization performance, achieving a higher mAP@0.5:0.95 (0.9738) compared to YOLOv5 (0.941). This shows that YOLOv8 produces tighter and more accurate bounding boxes at stricter IoU thresholds.

The Ensemble model using Weighted Box Fusion (WBF) achieved:

- Perfect detection accuracy (1.0)
- High average IoU of 0.9562

Although mAP@0.5 remained unchanged due to performance saturation on the small dataset, the ensemble improved bounding box stability and confidence robustness. The ensemble predictions were more consistent and reduced minor localization variations between the two models.

---

## Conclusion

1. Both YOLOv5 and YOLOv8 are highly effective for signature detection.
2. YOLOv8 provides better localization precision.
3. Ensemble learning enhances prediction stability and bounding box consistency.
4. On small, clean datasets, accuracy saturates quickly, limiting large numerical gains from ensemble methods.